In [ ]:
import numpy as np
import torch
from torch import nn
from torch.func import jvp
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

import os
from pathlib import Path
from tqdm import tqdm

from src import *

In [ ]:
torch.manual_seed(42)
jobs = (1, 2, 3)

In [ ]:
num_workers = 0 if os.name == "nt" else 24
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
metadata = {"sigma": SIGMA, "qfi_d": PHOTONS / (4 * SIGMA**2), "x": as_arr(x_coord)}

In [ ]:
def cfi_d(model, d):
    mu, dmu = jvp(model, (d,), (torch.ones_like(d),))
    fi = (dmu**2 / mu.clamp_min(1e-12)).sum(dim=1)
    return fi


qfi_d = PHOTONS / (4 * SIGMA**2)


def cfim(model, theta):
    dc = torch.zeros_like(theta)
    dc[:, 0] = 1

    dd = torch.zeros_like(theta)
    dd[:, 1] = 1

    mu, dmu_dc = jvp(model, (theta,), (dc,))
    _, dmu_dd = jvp(model, (theta,), (dd,))

    inv_mu = 1 / mu.clamp_min(1e-12)

    f_cc = (dmu_dc**2 * inv_mu).sum(dim=1)
    f_dd = (dmu_dd**2 * inv_mu).sum(dim=1)
    f_cd = (dmu_dc * dmu_dd * inv_mu).sum(dim=1)

    return torch.stack(
        (torch.stack((f_cc, f_cd), dim=-1), torch.stack((f_cd, f_dd), dim=-1)), dim=-2
    )


def qfim(d):
    f_dd = torch.full_like(d, qfi_d)

    f_cc = (
        PHOTONS
        / SIGMA**2
        * (1 - d**2 / (4 * SIGMA**2) * torch.exp(-(d**2) / (4 * SIGMA**2)))
    )

    f_cd = torch.zeros_like(d)

    return torch.stack(
        (torch.stack((f_cc, f_cd), dim=-1), torch.stack((f_cd, f_dd), dim=-1)), dim=-2
    )


def gaus_2psls(c, d):
    loc = torch.stack((c - d / 2, c + d / 2), dim=-1)
    return gaus(loc[..., None]) / 2**0.5

In [ ]:
class PhaseModOptim(nn.Module):
    def __init__(self, num_slms, d_only, bandwidth=0.1, init_std=0.1, learnable=True):
        super().__init__()
        self.d_only = d_only

        slms = []
        for _ in range(num_slms):
            slms.append(PSLM(bandwidth, init_std, learnable))
            slms.append(Lens())

        self.pmo_sys = nn.Sequential(*slms)

    def sample(self):
        self.detector = Detector("gaus")

    def direct(self):
        self.detector = Detector("direct")

    def forward(self, theta):
        d = theta if self.d_only else theta[:, 1]
        c = torch.zeros_like(d) if self.d_only else theta[:, 0]
        return self.detector(self.pmo_sys(gaus_2psls(c, d)))

    @property
    def phases(self):
        return torch.stack([c.phase for c in self.pmo_sys if isinstance(c, PSLM)])

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden=16, depth=3, activation=nn.ReLU(), d_only=True):
        super().__init__()
        self.d_only = d_only
        layers = [nn.Linear(X_SIZE, hidden), activation]

        for _ in range(depth - 2):
            layers += [nn.Linear(hidden, hidden), activation]

        self.backbone = nn.Sequential(*layers)

        self.head_c = (
            nn.Sequential(
                nn.Linear(hidden, int(hidden / 2)),
                activation,
                nn.Linear(int(hidden / 2), 1),
            )
            if not self.d_only
            else None
        )

        self.head_d = nn.Sequential(
            nn.Linear(hidden, int(hidden / 2)),
            activation,
            nn.Linear(int(hidden / 2), 1),
        )

    def forward(self, x):
        h = self.backbone(x.flatten(1))
        if self.d_only:
            d = self.head_d(h)
            return d
        else:
            c = self.head_c(h)
            d = self.head_d(h)
            return torch.cat((c, d), dim=1)

In [ ]:
class RandomCnD(Dataset):
    def __init__(self, n, c_range, d_range, unit=SIGMA):
        self.labels = torch.empty(n, 2)
        self.labels[:, 0] = torch.empty(n).uniform_(*c_range) * unit
        self.labels[:, 1] = torch.empty(n).uniform_(*d_range) * unit

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.labels[idx]

# ${\rm loss} = F(d)$

In [ ]:
if 1 in jobs:
    dataset = RandomCnD(int(2**20), c_range=(0, 0), d_range=(0, 3))
    loader = DataLoader(
        dataset, batch_size=int(2**10), shuffle=True, num_workers=num_workers
    )

In [ ]:
if 1 in jobs:
    results = {
        "2": {"phases": None, "cfi": None},
        "4": {"phases": None, "cfi": None},
    }

    epochs = 32
    small_d_reward = 2
    test_d = torch.linspace(0, 3 * SIGMA, 300, device=device)

In [ ]:
if 1 in jobs:
    for num_slms in (2, 4):
        model = PhaseModOptim(num_slms=num_slms, d_only=True).to(device)
        model.direct()
        model.train()

        optimizer = optim.Adam(model.parameters())
        history = []

        for epoch in range(epochs):
            total_loss = 0
            pbar = tqdm(loader, mininterval=10, desc=f"Epoch {epoch + 1}/{epochs}")

            for i, x_in in enumerate(pbar):
                d = x_in[:, 1].to(device)
                optimizer.zero_grad()

                weight = torch.exp(-small_d_reward * d / SIGMA)
                loss_cfi = (weight * (1 - cfi_d(model, d) / qfi_d)).mean()
                efficiency = model.detector.total.mean()
                loss = loss_cfi + (1 - efficiency)
                loss.backward()

                optimizer.step()

                total_loss += loss.item() * len(d)
                avg_loss = total_loss / ((i + 1) * loader.batch_size)

                pbar.set_postfix(
                    loss=f"{loss.item():.6f}",
                    avg=f"{avg_loss:.6f}",
                    eff=f"{efficiency:.6f}",
                )

            history.append(total_loss / len(loader.dataset))

        with torch.no_grad():
            results[str(num_slms)] = {
                "cfi": as_arr(cfi_d(model, test_d)),
                "phases": as_arr(model.phases),
                "history": np.asarray(history),
            }
        np.savez(
            results_dir / "single_fisher.npz",
            **metadata,
            test_d=as_arr(test_d),
            results=results,
        )

In [ ]:
if 1 in jobs:
    di_model = PhaseModOptim(num_slms=0, d_only=True, learnable=False).to(device)
    di_model.direct()
    with torch.no_grad():
        results["di"] = {"cfi": as_arr(cfi_d(di_model, test_d))}
    np.savez(
        results_dir / "single_fisher.npz",
        **metadata,
        test_d=as_arr(test_d),
        results=results,
    )

# ${\rm loss} = {\rm MSE}(\hat d)$

In [ ]:
if 2 in jobs:
    dataset = RandomCnD(int(2**20), c_range=(0, 0), d_range=(0, 3))
    loader = DataLoader(
        dataset, batch_size=int(2**10), shuffle=True, num_workers=num_workers
    )

In [ ]:
if 2 in jobs:
    results = {}
    test_d = torch.linspace(0, 3 * SIGMA, 300, device=device)
    num_samples = 10000

    epochs = 8
    criterion = nn.MSELoss(reduction="none")
    small_d_reward = 2

In [ ]:
if 2 in jobs:
    for num_slms in (0, 2, 4):
        pmo = PhaseModOptim(num_slms=num_slms, d_only=True, learnable=num_slms > 0).to(
            device
        )
        pmo.sample()

        estimator = MLP(d_only=True).to(device)
        model = nn.Sequential(pmo, estimator)
        model.train()

        optimizer = optim.Adam(model.parameters())
        history = []

        name = "di" if num_slms == 0 else str(num_slms)

        for epoch in range(epochs):
            total_loss = 0
            pbar = tqdm(loader, mininterval=10, desc=f"{name} | Epoch {epoch + 1}/{epochs}")

            for i, x_in in enumerate(pbar):
                d = x_in[:, 1].to(device)
                optimizer.zero_grad()

                pred = model(d).squeeze(1)

                weight = torch.exp(-small_d_reward * d / SIGMA)
                loss_mse = (weight * criterion(pred / SIGMA, d / SIGMA)).mean()
                efficiency = pmo.detector.total.mean()
                loss = loss_mse + (1 - efficiency)

                loss.backward()
                optimizer.step()

                total_loss += loss.item() * len(d)
                avg_loss = total_loss / ((i + 1) * loader.batch_size)

                pbar.set_postfix(
                    loss=f"{loss.item():.6f}",
                    mse=f"{loss_mse.item():.6f}",
                    avg=f"{avg_loss:.6f}",
                    eff=f"{efficiency:.6f}",
                )

            history.append(total_loss / len(loader.dataset))

        model.eval()
        mean_list, var_list = [], []
        with torch.no_grad():
            for d0 in test_d:
                d = torch.full((num_samples,), d0.item(), device=device)
                pred = model(d).squeeze(1)
                mean_list.append(pred.mean())
                var_list.append(pred.var(unbiased=True))

        results[name] = {
            "mean": as_arr(torch.stack(mean_list)),
            "var": as_arr(torch.stack(var_list)),
            "phases": as_arr(pmo.phases) if num_slms else None,
            "history": np.asarray(history),
        }
        np.savez(
            results_dir / "single_mse.npz",
            **metadata,
            test_d=as_arr(test_d),
            results=results,
        )

# ${\rm loss} = {\rm MSE}(\hat c)^{1-w}\,{\rm MSE}(\hat d)^w$

In [ ]:
if 3 in jobs:
    multi_results = {}
    di_model = PhaseModOptim(num_slms=0, d_only=False, learnable=False).to(device)
    di_model.direct()
    num_samples = 2000

In [ ]:
if 3 in jobs:
    for d_real in (0.1, 1, 2, 3):
        dataset = RandomCnD(
            int(2**20), c_range=(-0.1, 0.1), d_range=(d_real - 0.1, d_real + 0.1)
        )
        loader = DataLoader(
            dataset, batch_size=int(2**10), shuffle=True, num_workers=num_workers
        )

        d_reward = 0.9
        epochs = 32
        results = {}
        theta = torch.tensor([[0.0, d_real * SIGMA]], device=device)
        with torch.no_grad():
            multi_results[d_real] = {
                "theta": as_arr(theta[0]),
                "qfim": as_arr(qfim(theta[:, 1])[0]),
                "fim_di": as_arr(cfim(di_model, theta)[0]),
                "results": results,
            }

        for num_slms in [2, 4, 8, 12]:
            model = nn.Sequential(
                PhaseModOptim(num_slms=num_slms, d_only=False),
                MLP(d_only=False),
            ).to(device)
            model[0].sample()
            model.train()

            optimizer = optim.Adam(model.parameters())
            history = []

            for epoch in range(epochs):
                total_loss = 0
                pbar = tqdm(
                    loader, mininterval=10, desc=f"num_slms={num_slms} | Epoch {epoch + 1}/{epochs}"
                )

                for i, label in enumerate(pbar):
                    optimizer.zero_grad()
                    label = label.to(device)
                    c = label[:, 0]
                    d = label[:, 1]

                    pred = model(label)
                    efficiency = model[0].detector.total.mean()

                    loss_c = ((pred[:, 0] - c) / SIGMA).square().mean()
                    loss_d = ((pred[:, 1] - d) / SIGMA).square().mean()

                    eps = 1e-12
                    loss_est = (1 - d_reward) * torch.log(
                        loss_c + eps
                    ) + d_reward * torch.log(loss_d + eps)
                    loss = loss_est + (1 - efficiency)

                    loss.backward()
                    optimizer.step()

                    total_loss += loss.item() * len(label)
                    avg_loss = total_loss / ((i + 1) * loader.batch_size)

                    pbar.set_postfix(
                        loss=f"{loss.item():.6f}",
                        c=f"{loss_c.item():.6f}",
                        d=f"{loss_d.item():.6f}",
                        avg=f"{avg_loss:.6f}",
                        eff=f"{efficiency:.4f}",
                    )

                history.append(total_loss / len(loader.dataset))
            model.eval()
            with torch.no_grad():
                samples = (
                    as_arr(model(theta.repeat(num_samples, 1))) if d_real == 2 else None
                )
                model[0].direct()
                results[str(num_slms)] = {
                    "fim": as_arr(cfim(model[0], theta)[0]),
                    "phases": as_arr(model[0].phases),
                    "history": np.asarray(history),
                    "samples": samples,
                }
            np.savez(results_dir / "multi_mse.npz", **metadata, results=multi_results)